<a href="https://colab.research.google.com/github/DiyaRana7/Flyrank_ML/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked action approach

The playbook uses the ranked queue created by the baseline scoring approach.

Pages are prioritized for human review using observable content and search-performance signals available at the decision moment. The ranking is intended to answer: "Which pages should an editor review first?"

### Action labels

- `REVIEW_REFRESH` — the page has a higher baseline score and should receive human review for a possible content refresh.
- `MONITOR` — the page has a lower baseline score and does not receive immediate refresh priority.

### Reason codes

- `STALE_VISIBLE` — the page is at least 180 days old and has at least 500 impressions.
- `STALE` — the page is at least 180 days old but has lower visibility.
- `VISIBLE` — the page has at least 500 impressions but is not stale.
- `OTHER` — the page does not meet the main priority conditions.

The queue is a prioritization tool, not an automatic decision about whether a page should be changed.

In [18]:
import pandas as pd
import numpy as np
import os

# Make sure we are inside the repository
REPO_PATH = "/content/Flyrank_ML"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/DiyaRana7/Flyrank_ML.git

os.chdir(REPO_PATH)

# Load dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Working directory:", os.getcwd())
print("Dataset shape:", df.shape)

# -----------------------------
# Recreate ML-07 baseline queue
# -----------------------------

queue = df.copy()

# Start score at zero
queue["score"] = 0

# Staleness
queue["score"] += (
    queue["days_since_last_update"] >= 180
).astype(int) * 2

queue["score"] += (
    queue["days_since_last_update"] >= 365
).astype(int)

# Visibility
queue["score"] += (
    queue["impressions_90d"] >= 500
).astype(int) * 2

queue["score"] += (
    queue["impressions_90d"] >= 5000
).astype(int)

# Average position
queue["score"] += (
    queue["avg_position"] > 10
).astype(int)

# Reason codes
queue["reason_code"] = np.select(
    [
        (queue["days_since_last_update"] >= 180) &
        (queue["impressions_90d"] >= 500),

        queue["days_since_last_update"] >= 180,

        queue["impressions_90d"] >= 500
    ],
    [
        "STALE_VISIBLE",
        "STALE",
        "VISIBLE"
    ],
    default="OTHER"
)

# Action label
queue["action"] = np.where(
    queue["score"] >= 4,
    "REVIEW_REFRESH",
    "MONITOR"
)

# Confidence note
queue["confidence_note"] = np.select(
    [
        queue["score"] >= 6,
        queue["score"] >= 4
    ],
    [
        "Multiple strong priority signals",
        "Meets baseline review threshold"
    ],
    default="Lower priority based on baseline"
)

# Rank
queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

print("\nQueue created successfully.")
print("Total pages:", len(queue))

print("\nTop 20 ranked actions:")

display(
    queue[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "confidence_note"
        ]
    ].head(20)
)

Working directory: /content/Flyrank_ML
Dataset shape: (30000, 44)

Queue created successfully.
Total pages: 30000

Top 20 ranked actions:


,rank,content_id,score,reason_code,action,confidence_note
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
1,2,content_7368877ea310,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
2,3,content_1bfaa38ff26c,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
3,4,content_0a91db491d14,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
4,5,content_5feee3994adb,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
5,6,content_c2d929d83eaa,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals
6,7,content_b16bd7307b39,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
7,8,content_fe16a55cd13d,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
8,9,content_ecb6215e79fd,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold
9,10,content_928af3e22c80,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is designed to help content editors prioritize pages for human review.

The ranked queue provides a consistent starting point for deciding which pages should be reviewed first for a possible content refresh. The reason code explains the main observable signal behind the recommendation.

The output is intended for decision-support and prioritization, not automatic content changes.

### Limits

The score does not prove that a page needs a refresh or that a refresh will improve search performance.

The signals describe observed content age and search-performance patterns. They cannot establish causality or explain Google's ranking decisions.

The recommendations may also become less reliable when data coverage changes, when the underlying data distribution shifts, or when the content context differs substantially from the data used to create the baseline.

Human review is required before taking any content action.

In [19]:
# Check the coverage and current recommendation distribution

print("Total pages in queue:", len(queue))
print("Unique content pages:", queue["content_id"].nunique())

print("\nAction distribution:")
display(
    queue["action"]
    .value_counts()
    .to_frame("count")
)

print("\nReason-code distribution:")
display(
    queue["reason_code"]
    .value_counts()
    .to_frame("count")
)

print("\nMissing values in key recommendation fields:")

display(
    queue[
        [
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "avg_position"
        ]
    ].isnull().sum().to_frame("missing_count")
)

Total pages in queue: 30000
Unique content pages: 30000

Action distribution:


,count
action,
MONITOR,27405
REVIEW_REFRESH,2595



Reason-code distribution:


,count
reason_code,
VISIBLE,16709
OTHER,13117
STALE,157
STALE_VISIBLE,17



Missing values in key recommendation fields:


,missing_count
score,0
reason_code,0
action,0
days_since_last_update,0
impressions_90d,0
avg_position,0


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every `REVIEW_REFRESH` recommendation must be reviewed by a person before any content change is made.

The reviewer should check:

- whether the page is still relevant to its audience;
- whether the information is outdated, incomplete, or inaccurate;
- whether the page already performs well despite the signals;
- whether the search visibility is meaningful for this page;
- whether there are editorial, business, legal, or other reasons not to change it;
- whether a refresh is appropriate for the page's content type.

The reason code should be used as a starting point for the review, not as a final decision.

### No-go list

The system should NOT automatically:

- rewrite or publish content;
- delete or merge pages;
- change titles or metadata;
- claim that a refresh will improve rankings or traffic;
- make claims about Google's ranking algorithm;
- override editorial or subject-matter expertise;
- make decisions where the available data is incomplete or clearly unreliable.

The playbook is a prioritization and decision-support tool. Final content actions remain human decisions.

In [20]:
# Show the highest-priority pages that require human review

review_queue = queue[
    queue["action"] == "REVIEW_REFRESH"
].copy()

print("Pages requiring human review:", len(review_queue))

display(
    review_queue[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "confidence_note",
            "days_since_last_update",
            "impressions_90d",
            "avg_position"
        ]
    ].head(10)
)

Pages requiring human review: 2595


,rank,content_id,score,reason_code,confidence_note,days_since_last_update,impressions_90d,avg_position
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,Multiple strong priority signals,194,61678,19.7
1,2,content_7368877ea310,6,STALE_VISIBLE,Multiple strong priority signals,194,59472,24.8
2,3,content_1bfaa38ff26c,6,STALE_VISIBLE,Multiple strong priority signals,194,25715,22.2
3,4,content_0a91db491d14,6,STALE_VISIBLE,Multiple strong priority signals,193,13299,10.5
4,5,content_5feee3994adb,6,STALE_VISIBLE,Multiple strong priority signals,194,7812,39.0
5,6,content_c2d929d83eaa,6,STALE_VISIBLE,Multiple strong priority signals,193,7558,17.9
6,7,content_b16bd7307b39,5,STALE_VISIBLE,Meets baseline review threshold,194,4590,31.0
7,8,content_fe16a55cd13d,5,STALE_VISIBLE,Meets baseline review threshold,194,4556,16.4
8,9,content_ecb6215e79fd,5,STALE_VISIBLE,Meets baseline review threshold,194,4429,25.3
9,10,content_928af3e22c80,5,STALE_VISIBLE,Meets baseline review threshold,193,1697,15.8


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 4. Monitoring / retrain triggers

The action playbook should be monitored because search performance and content conditions can change over time.

I would review the recommendations periodically using these signals:

- The distribution of baseline scores changes substantially.
- The proportion of pages receiving `REVIEW_REFRESH` changes substantially.
- The observed declining rate changes from the rate used during validation.
- Model performance, especially F1 and ROC-AUC, falls on newly observed data.
- The most important features change substantially compared with the validated model.
- New content types or traffic patterns appear that were not represented in the training data.

A model retrain should be considered when performance on a recent evaluation window is consistently weaker than the validated result, or when the underlying data distribution has materially changed.

These are monitoring and decision-support triggers, not automatic production actions. Any refresh recommendation should still be reviewed by a human.

In [21]:
# Basic monitoring summary for the current recommendation queue

print("Monitoring summary")
print("------------------")

print("Total pages:", len(queue))

print("\nAction distribution:")
display(queue["action"].value_counts())

print("\nScore distribution:")
display(queue["score"].value_counts().sort_index())

print("\nReason-code distribution:")
display(queue["reason_code"].value_counts())

print("\nAverage key signals:")
print("Days since update:", round(queue["days_since_last_update"].mean(), 2))
print("Impressions (90d):", round(queue["impressions_90d"].mean(), 2))
print("Average position:", round(queue["avg_position"].mean(), 2))

print("\nMonitoring note:")
print(
    "These values provide a reference point for future monitoring. "
    "Large changes in action rates, score distribution, or feature distributions "
    "should trigger a review of the baseline/model."
)

Monitoring summary
------------------
Total pages: 30000

Action distribution:


,count
action,
MONITOR,27405
REVIEW_REFRESH,2595



Score distribution:


,count
score,
0,6516
1,6601
2,4097
3,10191
4,2581
5,8
6,6



Reason-code distribution:


,count
reason_code,
VISIBLE,16709
OTHER,13117
STALE,157
STALE_VISIBLE,17



Average key signals:
Days since update: 46.1
Impressions (90d): 5200.37
Average position: 16.34

Monitoring note:
These values provide a reference point for future monitoring. Large changes in action rates, score distribution, or feature distributions should trigger a review of the baseline/model.


## 5. Exports for the paper

The validated ranked queue is exported to `work/outputs/` so that the research paper can reuse the same recommendations produced by this notebook.

The export contains the page-level rank, score, reason code, action, and supporting observable signals. The queue is intended for human review and is not an automated production decision system.

The CSV is regenerated by the notebook rather than treated as a manually edited source of truth.

In [22]:
import os

# Create the output directory
os.makedirs("work/outputs", exist_ok=True)

# Use the validated queue if it already exists.
# Otherwise recreate the baseline queue from the current dataframe.
if "queue" not in globals():

    baseline = df.copy()
    baseline["score"] = 0

    # Staleness signals
    baseline["score"] += (
        baseline["days_since_last_update"] >= 180
    ).astype(int) * 2

    baseline["score"] += (
        baseline["days_since_last_update"] >= 365
    ).astype(int)

    # Visibility signals
    baseline["score"] += (
        baseline["impressions_90d"] >= 500
    ).astype(int) * 2

    baseline["score"] += (
        baseline["impressions_90d"] >= 5000
    ).astype(int)

    # Position signal
    baseline["score"] += (
        baseline["avg_position"] > 10
    ).astype(int)

    # Reason codes
    baseline["reason_code"] = np.select(
        [
            (baseline["days_since_last_update"] >= 180) &
            (baseline["impressions_90d"] >= 500),

            baseline["days_since_last_update"] >= 180,

            baseline["impressions_90d"] >= 500
        ],
        [
            "STALE_VISIBLE",
            "STALE",
            "VISIBLE"
        ],
        default="OTHER"
    )

    # Action
    baseline["action"] = np.where(
        baseline["score"] >= 4,
        "REVIEW_REFRESH",
        "MONITOR"
    )

    # Confidence note
    baseline["confidence_note"] = np.select(
        [
            baseline["score"] >= 6,
            baseline["score"] >= 4
        ],
        [
            "Multiple strong priority signals",
            "Meets baseline review threshold"
        ],
        default="Lower priority based on baseline"
    )

    # Rank
    queue = baseline.sort_values(
        ["score", "impressions_90d"],
        ascending=[False, False]
    ).reset_index(drop=True)

    queue["rank"] = range(1, len(queue) + 1)


# Select safe fields for the paper
export_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

export_columns = [
    col for col in export_columns
    if col in queue.columns
]

paper_queue = queue[export_columns].copy()

output_path = "work/outputs/action_playbook_queue.csv"

paper_queue.to_csv(output_path, index=False)

print("Export completed successfully.")
print("Path:", output_path)
print("Rows:", len(paper_queue))
print("Columns:", paper_queue.columns.tolist())

display(paper_queue.head(20))

Export completed successfully.
Path: work/outputs/action_playbook_queue.csv
Rows: 30000
Columns: ['rank', 'content_id', 'score', 'reason_code', 'action', 'confidence_note', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


,rank,content_id,score,reason_code,action,confidence_note,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,61678,19.7,0.15
1,2,content_7368877ea310,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,59472,24.8,0.13
2,3,content_1bfaa38ff26c,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,25715,22.2,0.23
3,4,content_0a91db491d14,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,193,13299,10.5,0.49
4,5,content_5feee3994adb,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,7812,39.0,0.01
5,6,content_c2d929d83eaa,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,193,7558,17.9,0.20
6,7,content_b16bd7307b39,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4590,31.0,0.00
7,8,content_fe16a55cd13d,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4556,16.4,0.33
8,9,content_ecb6215e79fd,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,194,4429,25.3,0.38
9,10,content_928af3e22c80,5,STALE_VISIBLE,REVIEW_REFRESH,Meets baseline review threshold,193,1697,15.8,0.12


In [23]:
print("File exists:", os.path.exists("work/outputs/action_playbook_queue.csv"))
print("Export rows:", len(pd.read_csv("work/outputs/action_playbook_queue.csv")))

File exists: True
Export rows: 30000


## Self-check

- [x] Every section is filled with both markdown reasoning and supporting code.
- [x] The notebook runs from top to bottom without errors after restarting the runtime and running all cells.
- [x] The ranked action queue is exported to `work/outputs/`.
- [x] The recommendations are intended for human review and decision support, not automatic production actions.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No client names, private queries, credentials, or other client-identifying information are included.
- [x] Monitoring and retraining triggers are stated explicitly.
- [x] The exported queue can be regenerated from the notebook.
- [x] The notebook is saved under `work/notebooks/w07_action_playbook.ipynb`.

In [24]:
import os
import pandas as pd

print("=== ML-10 FINAL CHECK ===")

print("Notebook/output directory exists:",
      os.path.exists("work/outputs"))

output_path = "work/outputs/action_playbook_queue.csv"

print("Queue export exists:",
      os.path.exists(output_path))

if os.path.exists(output_path):
    exported = pd.read_csv(output_path)

    print("Export rows:", len(exported))
    print("Export columns:", exported.columns.tolist())

    print("\nAction distribution:")
    print(exported["action"].value_counts())

    print("\nTop 5 ranked actions:")
    display(exported.head(5))

print("\nML-10 playbook export check complete.")

=== ML-10 FINAL CHECK ===
Notebook/output directory exists: True
Queue export exists: True
Export rows: 30000
Export columns: ['rank', 'content_id', 'score', 'reason_code', 'action', 'confidence_note', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']

Action distribution:
action
MONITOR           27405
REVIEW_REFRESH     2595
Name: count, dtype: int64

Top 5 ranked actions:


,rank,content_id,score,reason_code,action,confidence_note,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,61678,19.7,0.15
1,2,content_7368877ea310,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,59472,24.8,0.13
2,3,content_1bfaa38ff26c,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,25715,22.2,0.23
3,4,content_0a91db491d14,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,193,13299,10.5,0.49
4,5,content_5feee3994adb,6,STALE_VISIBLE,REVIEW_REFRESH,Multiple strong priority signals,194,7812,39.0,0.01



ML-10 playbook export check complete.
